In [0]:
%run /Users/mathisneha2004@gmail.com/config/Pipeline_Config

In [0]:
dbutils.widgets.text(
    "partition_num", "0", "Partition Number (0-3)"
)
partition_num = dbutils.widgets.get("partition_num")
print(f"✓ Processing partition: {partition_num}")

In [0]:
from datetime import datetime
bronze_start = datetime.now()
print(f"{'='*50}")
print(f"BRONZE LAYER AUDIT LOG")
print(f"{'='*50}")
print(f"Start Time  : {bronze_start}")
print(f"Partition   : {partition_num}")
print(f"Source      : claims_datasetnew_part_{partition_num}")
print(f"{'='*50}")

## Setup: Catalog and Bronze Schema

Creating Unity Catalog infrastructure for healthcare claims data

In [0]:
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    print("✓ Catalog healthcare_claims_data created successfully")
except Exception as e:
    print(f"✗ Failed to create catalog: {str(e)}")
    raise

In [0]:
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")
    print("✓ Schema healthcare_claims_data.bronze created successfully")
except Exception as e:
    print(f"✗ Failed to create schema: {str(e)}")
    raise

## Step 1: Read Data from Bronze Tables

**Claims**: Reading from pre-existing `claims_dataset1` table (2M records subset)
**Hospital**: Reading from S3

In [0]:
try:
    claims_raw = spark.table(
        f"{BRONZE_PARTITION_TABLE}_{partition_num}"
    )
    print(f"Claims: {len(claims_raw.columns)} columns")
    print("✓ Claims raw data loaded successfully")
except Exception as e:
    print(f"✗ Failed to read claims data: {str(e)}")
    raise

In [0]:
try:
    display(claims_raw.limit(50))
except Exception as e:
    print(f"✗ Failed to display claims sample: {str(e)}")
    raise

In [0]:
try:
    hospital_raw = spark.read.format("csv") \
        .option("header", "True") \
        .option("inferSchema", "True") \
        .load(S3_HOSPITAL_PATH)
    print(f"Hospital: {len(hospital_raw.columns)} columns")
    print("✓ Hospital raw data loaded successfully")
except Exception as e:
    print(f"✗ Failed to read hospital data: {str(e)}")
    raise

In [0]:
try:
    display(hospital_raw.limit(5))
except Exception as e:
    print(f"✗ Failed to display hospital sample: {str(e)}")
    raise

## Step 2: Define Hospital Schema and Re-read

In [0]:
try:
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType

    claims_schema = StructType([
        StructField("DESYNPUF_ID", StringType(), True),
        StructField("BENE_BIRTH_DT", IntegerType(), True),
        StructField("BENE_DEATH_DT", IntegerType(), True),
        StructField("BENE_SEX_IDENT_CD", IntegerType(), True),
        StructField("BENE_RACE_CD", IntegerType(), True),
        StructField("BENE_ESRD_IND", StringType(), True),
        StructField("SP_STATE_CODE", IntegerType(), True),
        StructField("BENE_COUNTY_CD", IntegerType(), True),
        StructField("BENE_HI_CVRAGE_TOT_MONS", IntegerType(), True),
        StructField("BENE_SMI_CVRAGE_TOT_MONS", IntegerType(), True),
        StructField("BENE_HMO_CVRAGE_TOT_MONS", IntegerType(), True),
        StructField("PLAN_CVRG_MOS_NUM", IntegerType(), True),
        StructField("SP_ALZHDMTA", IntegerType(), True),
        StructField("SP_CHF", IntegerType(), True),
        StructField("SP_CHRNKIDN", IntegerType(), True),
        StructField("SP_CNCR", IntegerType(), True),
        StructField("SP_COPD", IntegerType(), True),
        StructField("SP_DEPRESSN", IntegerType(), True),
        StructField("SP_DIABETES", IntegerType(), True),
        StructField("SP_ISCHMCHT", IntegerType(), True),
        StructField("SP_OSTEOPRS", IntegerType(), True),
        StructField("SP_RA_OA", IntegerType(), True),
        StructField("SP_STRKETIA", IntegerType(), True),
        StructField("MEDREIMB_IP", DoubleType(), True),
        StructField("BENRES_IP", DoubleType(), True),
        StructField("PPPYMT_IP", DoubleType(), True),
        StructField("MEDREIMB_OP", DoubleType(), True),
        StructField("BENRES_OP", DoubleType(), True),
        StructField("PPPYMT_OP", DoubleType(), True),
        StructField("MEDREIMB_CAR", DoubleType(), True),
        StructField("BENRES_CAR", DoubleType(), True),
        StructField("PPPYMT_CAR", DoubleType(), True),
        StructField("CLM_ID", LongType(), True),
        StructField("CLM_FROM_DT", IntegerType(), True),
        StructField("CLM_THRU_DT", IntegerType(), True),
        StructField("ICD9_DGNS_CD_1", StringType(), True),
        StructField("PRF_PHYSN_NPI_1", DoubleType(), True),
        StructField("HCPCS_CD_1", StringType(), True),
        StructField("LINE_NCH_PMT_AMT_1", DoubleType(), True),
        StructField("LINE_BENE_PTB_DDCTBL_AMT_1", DoubleType(), True),
        StructField("LINE_COINSRNC_AMT_1", DoubleType(), True),
        StructField("LINE_PRCSG_IND_CD_1", StringType(), True),
        StructField("LINE_ICD9_DGNS_CD_1", StringType(), True),
        StructField("part_num", IntegerType(), True),
        StructField("partition_id", IntegerType(), True)
    ])

    print(f"Claims schema: {len(claims_schema.fields)} fields")
    print("✓ Claims schema defined successfully")
except Exception as e:
    print(f"✗ Failed to define claims schema: {str(e)}")
    raise

In [0]:
try:
    from pyspark.sql.functions import col
    
    claims_raw_df = spark.table(
        f"{BRONZE_PARTITION_TABLE}_{partition_num}"
    )
    
    # Apply claims_schema by casting each column
    claims_df = claims_raw_df.select([
        col("DESYNPUF_ID").cast("string"),
        col("BENE_BIRTH_DT").cast("int"),
        col("BENE_DEATH_DT").cast("int"),
        col("BENE_SEX_IDENT_CD").cast("int"),
        col("BENE_RACE_CD").cast("int"),
        col("BENE_ESRD_IND").cast("string"),
        col("SP_STATE_CODE").cast("int"),
        col("BENE_COUNTY_CD").cast("int"),
        col("BENE_HI_CVRAGE_TOT_MONS").cast("int"),
        col("BENE_SMI_CVRAGE_TOT_MONS").cast("int"),
        col("BENE_HMO_CVRAGE_TOT_MONS").cast("int"),
        col("PLAN_CVRG_MOS_NUM").cast("int"),
        col("SP_ALZHDMTA").cast("int"),
        col("SP_CHF").cast("int"),
        col("SP_CHRNKIDN").cast("int"),
        col("SP_CNCR").cast("int"),
        col("SP_COPD").cast("int"),
        col("SP_DEPRESSN").cast("int"),
        col("SP_DIABETES").cast("int"),
        col("SP_ISCHMCHT").cast("int"),
        col("SP_OSTEOPRS").cast("int"),
        col("SP_RA_OA").cast("int"),
        col("SP_STRKETIA").cast("int"),
        col("MEDREIMB_IP").cast("double"),
        col("BENRES_IP").cast("double"),
        col("PPPYMT_IP").cast("double"),
        col("MEDREIMB_OP").cast("double"),
        col("BENRES_OP").cast("double"),
        col("PPPYMT_OP").cast("double"),
        col("MEDREIMB_CAR").cast("double"),
        col("BENRES_CAR").cast("double"),
        col("PPPYMT_CAR").cast("double"),
        col("CLM_ID").cast("long"),
        col("CLM_FROM_DT").cast("int"),
        col("CLM_THRU_DT").cast("int"),
        col("ICD9_DGNS_CD_1").cast("string"),
        col("PRF_PHYSN_NPI_1").cast("double"),
        col("HCPCS_CD_1").cast("string"),
        col("LINE_NCH_PMT_AMT_1").cast("double"),
        col("LINE_BENE_PTB_DDCTBL_AMT_1").cast("double"),
        col("LINE_COINSRNC_AMT_1").cast("double"),
        col("LINE_PRCSG_IND_CD_1").cast("string"),
        col("LINE_ICD9_DGNS_CD_1").cast("string")
    ])
    
    print(f"Claims DataFrame: {len(claims_df.columns)} columns")
    print("✓ Claims DataFrame created successfully with schema applied")
except Exception as e:
    print(f"✗ Failed to create claims DataFrame: {str(e)}")
    raise

In [0]:
try:
    display(claims_df.limit(5))
except Exception as e:
    print(f"✗ Failed to display claims DataFrame: {str(e)}")
    raise

In [0]:
try:
    hospital_schema = StructType([
        StructField("Rndrng_Prvdr_CCN", StringType(), True),
        StructField("Rndrng_Prvdr_Org_Name", StringType(), True),
        StructField("Rndrng_Prvdr_City", StringType(), True),
        StructField("Rndrng_Prvdr_St", StringType(), True),
        StructField("Rndrng_Prvdr_State_FIPS", StringType(), True),
        StructField("Rndrng_Prvdr_Zip5", StringType(), True),
        StructField("Rndrng_Prvdr_State_Abrvtn", StringType(), True),
        StructField("Rndrng_Prvdr_RUCA", StringType(), True),
        StructField("Rndrng_Prvdr_RUCA_Desc", StringType(), True),
        StructField("DRG_Cd", StringType(), True),
        StructField("DRG_Desc", StringType(), True),
        StructField("Tot_Dschrgs", IntegerType(), True),
        StructField("Avg_Submtd_Cvrd_Chrg", DoubleType(), True),
        StructField("Avg_Tot_Pymt_Amt", DoubleType(), True),
        StructField("Avg_Mdcr_Pymt_Amt", DoubleType(), True)
    ])

    print(f"Hospital schema: {len(hospital_schema.fields)} fields")
    print("✓ Hospital schema defined successfully")
except Exception as e:
    print(f"✗ Failed to define hospital schema: {str(e)}")
    raise

In [0]:
try:
    hospital_df = spark.read.format("csv") \
        .option("header", "True") \
        .schema(hospital_schema) \
        .load(S3_HOSPITAL_PATH)
    print(f"Hospital DataFrame: {len(hospital_df.columns)} columns")
    print("✓ Hospital DataFrame created successfully")
except Exception as e:
    print(f"✗ Failed to create hospital DataFrame: {str(e)}")
    raise

In [0]:
try:
    display(hospital_df.limit(5))
except Exception as e:
    print(f"✗ Failed to display hospital DataFrame: {str(e)}")
    raise

## Step 3: Basic EDA - Claims Dataset

In [0]:
try:
    claims_df.printSchema()
    print("✓ Claims schema printed successfully")
except Exception as e:
    print(f"✗ Failed to print claims schema: {str(e)}")
    raise

In [0]:
try:
    total_claims = claims_df.count()
    print(f"Total Records: {total_claims:,}")
    print("✓ Claims count completed successfully")
except Exception as e:
    print(f"✗ Failed to count claims: {str(e)}")
    raise

In [0]:
try:
    from pyspark.sql.functions import col, count, when

    null_counts = claims_df.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in claims_df.columns
    ])

    display(null_counts)
    print("✓ Claims null check completed successfully")
except Exception as e:
    print(f"✗ Failed to check claims nulls: {str(e)}")
    raise

In [0]:
try:
    total = claims_df.count()
    unique = claims_df.select("CLM_ID").distinct().count()
    duplicates = total - unique

    print(f"Total: {total:,} | Unique CLM_ID: {unique:,} | Duplicates: {duplicates:,}")
    print("✓ Claims duplicate check completed successfully")
except Exception as e:
    print(f"✗ Failed to check claims duplicates: {str(e)}")
    raise

## Step 3: Basic EDA - Hospital Dataset

In [0]:
try:
    hospital_df.printSchema()
    print("✓ Hospital schema printed successfully")
except Exception as e:
    print(f"✗ Failed to print hospital schema: {str(e)}")
    raise

In [0]:
try:
    total_hospital = hospital_df.count()
    print(f"Total Records: {total_hospital:,}")
    print("✓ Hospital count completed successfully")
except Exception as e:
    print(f"✗ Failed to count hospital records: {str(e)}")
    raise

In [0]:
try:
    hospital_null_counts = hospital_df.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in hospital_df.columns
    ])

    display(hospital_null_counts)
    print("✓ Hospital null check completed successfully")
except Exception as e:
    print(f"✗ Failed to check hospital nulls: {str(e)}")
    raise

In [0]:
try:
    total_h = hospital_df.count()
    unique_h = hospital_df.select("Rndrng_Prvdr_CCN", "DRG_Cd").distinct().count()
    duplicates_h = total_h - unique_h

    print(f"Total: {total_h:,} | Unique: {unique_h:,} | Duplicates: {duplicates_h:,}")
    print("✓ Hospital duplicate check completed successfully")
except Exception as e:
    print(f"✗ Failed to check hospital duplicates: {str(e)}")
    raise

## Step 4: Add Metadata Columns

In [0]:
try:
    from pyspark.sql.functions import current_timestamp, current_date, lit

    claims_bronze = claims_df.withColumn("load_tmst", current_timestamp()) \
        .withColumn("load_date", current_date()) \
        .withColumn("source_id", lit(SOURCE_ID_CLAIMS))

    print(f"Claims: {len(claims_bronze.columns)} columns (with metadata)")
    print("✓ Metadata added to claims successfully")
except Exception as e:
    print(f"✗ Failed to add metadata to claims: {str(e)}")
    raise

In [0]:
try:
    hospital_bronze = hospital_df.withColumn("load_tmst", current_timestamp()) \
        .withColumn("load_date", current_date()) \
        .withColumn("source_id", lit(SOURCE_ID_HOSPITAL))

    print(f"Hospital: {len(hospital_bronze.columns)} columns (with metadata)")
    print("✓ Metadata added to hospital successfully")
except Exception as e:
    print(f"✗ Failed to add metadata to hospital: {str(e)}")
    raise

## Step 5: Save Bronze Tables

In [0]:
try:
    claims_bronze.write.format("delta") \
        .mode(BRONZE_WRITE_MODE) \
        .saveAsTable(BRONZE_CLAIMS_TABLE)

    print(f"✓ Claims saved to {BRONZE_CLAIMS_TABLE}")
except Exception as e:
    print(f"✗ Failed to save claims to bronze: {str(e)}")
    raise

In [0]:
try:
    hospital_bronze.write.format("delta") \
        .mode(BRONZE_WRITE_MODE) \
        .saveAsTable(BRONZE_HOSPITAL_TABLE)

    print(f"✓ Hospital saved to {BRONZE_HOSPITAL_TABLE}")
except Exception as e:
    print(f"✗ Failed to save hospital to bronze: {str(e)}")
    raise

## Step 6: Verification

In [0]:
try:
    display(spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{BRONZE_SCHEMA}"))
    print("✓ Bronze tables listed successfully")
except Exception as e:
    print(f"✗ Failed to list bronze tables: {str(e)}")
    raise

In [0]:
try:
    claims_verify = spark.table(BRONZE_CLAIMS_TABLE)
    print(f"Claims: {claims_verify.count():,} records | {len(claims_verify.columns)} columns")
    print(f"Metadata columns: {[c for c in claims_verify.columns if c in ['load_tmst', 'load_date', 'source_id']]}")
    display(claims_verify.select("DESYNPUF_ID", "CLM_ID", "load_tmst", "load_date", "source_id").limit(5))
    print("✓ Claims table verified successfully")
except Exception as e:
    print(f"✗ Failed to verify claims table: {str(e)}")
    raise

In [0]:
try:
    hospital_verify = spark.table(BRONZE_HOSPITAL_TABLE)
    print(f"Hospital: {hospital_verify.count():,} records | {len(hospital_verify.columns)} columns")
    print(f"Metadata columns: {[c for c in hospital_verify.columns if c in ['load_tmst', 'load_date', 'source_id']]}")
    display(hospital_verify.select("Rndrng_Prvdr_CCN", "Rndrng_Prvdr_Org_Name", "load_tmst", "load_date", "source_id").limit(5))
    print("✓ Hospital table verified successfully")
except Exception as e:
    print(f"✗ Failed to verify hospital table: {str(e)}")
    raise

In [0]:
bronze_end = datetime.now()
bronze_duration = (bronze_end - bronze_start).seconds

claims_final = spark.table(BRONZE_CLAIMS_TABLE).count()

hospital_final = spark.table(BRONZE_HOSPITAL_TABLE).count()

print(f"{'='*50}")
print(f"BRONZE AUDIT SUMMARY")
print(f"{'='*50}")
print(f"Partition          : {partition_num}")
print(f"Claims Loaded      : {claims_final:,}")
print(f"Hospital Loaded    : {hospital_final:,}")
print(f"Start Time         : {bronze_start}")
print(f"End Time           : {bronze_end}")
print(f"Duration           : {bronze_duration} seconds")
print(f"Status             : SUCCESS")
print(f"{'='*50}")